In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/student_depression_ml.csv")

df_ml.head()

,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress,Financial Stress__missing,Gender_male,City_agra,City_ahmedabad,City_bangalore,City_bhavna,City_bhopal,City_chennai,City_city,City_delhi,City_faridabad,City_gaurav,City_ghaziabad,City_harsh,City_harsha,City_hyderabad,City_indore,City_jaipur,City_kalyan,City_kanpur,City_khaziabad,City_kibara,City_kolkata,City_less delhi,City_less than 5 kalyan,City_lucknow,City_ludhiana,City_m.com,City_m.tech,City_me,City_meerut,City_mihir,City_mira,City_mumbai,City_nagpur,City_nalini,City_nalyan,City_nandini,City_nashik,City_patna,City_pune,City_rajkot,City_rashi,City_reyansh,City_saanvi,City_srinagar,City_surat,City_thane,City_vaanya,City_vadodara,City_varanasi,City_vasai-virar,City_visakhapatnam,Profession_chef,Profession_civil engineer,Profession_content writer,Profession_digital marketer,Profession_doctor,Profession_educational consultant,Profession_entrepreneur,Profession_lawyer,Profession_manager,Profession_pharmacist,Profession_student,Profession_teacher,Profession_ux/ui designer,Sleep Duration_7-8 hours,Sleep Duration_Less than 5 hours,Sleep Duration_More than 8 hours,Sleep Duration_Others,Dietary Habits_Moderate,Dietary Habits_Others,Dietary Habits_Unhealthy,Degree_b.com,Degree_b.ed,Degree_b.pharm,Degree_b.tech,Degree_ba,Degree_bba,Degree_bca,Degree_be,Degree_bhm,Degree_bsc,Degree_class 12,Degree_llb,Degree_llm,Degree_m.com,Degree_m.ed,Degree_m.pharm,Degree_m.tech,Degree_ma,Degree_mba,Degree_mbbs,Degree_mca,Degree_md,Degree_me,Degree_mhm,Degree_msc,Degree_others,Degree_phd,Have you ever had suicidal thoughts ?_yes,Family History of Mental Illness_yes,Depression_yes
0,33,5,0,8,2,0,3,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
1,24,2,0,5,5,0,3,2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,31,3,0,7,5,0,9,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,28,3,0,5,2,0,4,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
4,25,4,0,8,3,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0


In [3]:
# Convert boolean columns to integers (if any exist)
# This ensures all predictors are numeric and compatible with sklearn/XGBoost
bool_cols = df_ml.select_dtypes(include=["bool"]).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

# If some columns were loaded as object but contain only True/False strings,
# convert them to 0/1 as well
for col in df_ml.columns:
    if df_ml[col].dtype == "object":
        unique_vals = set(df_ml[col].dropna().astype(str).unique())
        if unique_vals.issubset({"True", "False"}):
            df_ml[col] = df_ml[col].map({"True": 1, "False": 0})

# Check resulting data types
df_ml.dtypes.value_counts()

int64    111
Name: count, dtype: int64

In [4]:
# Define feature matrix X and target vector y
# Depression_yes is the binary target variable -removed from X
X = df_ml.drop(columns=["Depression_yes"])
y = df_ml["Depression_yes"]

# Check dimensions of predictors and target
print(X.shape, y.shape)

# Check for missing values in the target should be 0
print("Missing values in target:", y.isna().sum())

# Display class distribution, check balance
print("\nTarget class distribution:")
print(y.value_counts())

print("\nTarget class distribution (normalized):")
print(y.value_counts(normalize=True))

(27901, 110) (27901,)
Missing values in target: 0

Target class distribution:
Depression_yes
1    16336
0    11565
Name: count, dtype: int64

Target class distribution (normalized):
Depression_yes
1    0.585499
0    0.414501
Name: proportion, dtype: float64


In [5]:
# Split the dataset into training and test sets
# test_size=0.2 means 80% training, 20% testing
# random_state ensures reproducibility
# stratify=y preserves class distribution in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display shapes of training and test sets
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Display class distribution in the training set
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

# Display class distribution in the test set
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train shape: (22320, 110)
Test shape: (5581, 110)

Train class distribution:
Depression_yes
1    0.585484
0    0.414516
Name: proportion, dtype: float64

Test class distribution:
Depression_yes
1    0.585558
0    0.414442
Name: proportion, dtype: float64
